In [ ]:
# 既にインストールされていれば不要
!pip install quri-parts
!pip install "quri-parts[qulacs]"

In [2]:
import math
N = 21
a = 2
L = 5 # Nを表すのに必要な量子ビット数
n_reg_qubits = 8 # 補助量子ビットの数. 精度を保証するには2L+1が必要
n_qubits = n_reg_qubits + L # 総量子ビット数

print(f"a = {a}, N = {N}, L = {L}, n_reg_qubits = {n_reg_qubits}")
a_N_gcd = math.gcd(a, N)
print("a, N の最大公約数:", a_N_gcd)
assert a_N_gcd == 1 # N と a が互いに素か確認

a = 2, N = 21, L = 5, n_reg_qubits = 8
a, N の最大公約数: 1


In [3]:
def calculate_period(a, N):
    """ a^r mod N = 1 を満たす最小のrを求める"""
    r = 1
    while True:
        if pow(a, r, N) == 1:
            return r
        r += 1
        if r > N: # 無限ループ防止
            return None

print(f"古典計算による位数: r = {calculate_period(a, N)}")

古典計算による位数: r = 6


In [4]:
import numpy as np
# (U_a)^{power} |x> = |a^{power} x mod N>を実現する
# L 量子ビットユニタリ行列を生成
def create_U_a(a: int, N: int, L: int, power:int = 1) -> np.ndarray:
    U_a_matrix = np.zeros((2**L, 2**L), dtype=float)

    # N 未満の整数 x についてはU|x> = |a*x mod N>
    for x in range(N):
        res = (pow(a, power) * x) % N
        U_a_matrix[res, x] = 1.0 # 行:出力状態, 列:入力状態

    # N 以上の整数については恒等変換とする
    for x in range(N, 2**L):
        U_a_matrix[x, x] = 1.0

    return U_a_matrix

print(create_U_a(7, 15, 4, power=1))

[[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]


In [5]:
eigs_U_a, _ = np.linalg.eig(create_U_a(a, N, L))
phases = np.angle(eigs_U_a)/(2 * np.pi)
print(phases)

[ 0.5         0.33333333 -0.33333333  0.16666667 -0.16666667  0.
  0.33333333 -0.33333333  0.          0.33333333 -0.33333333  0.
  0.          0.5         0.5         0.33333333 -0.33333333  0.16666667
 -0.16666667  0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.        ]


In [6]:
from quri_parts.core.state import quantum_state
from quri_parts.circuit import QuantumCircuit
from quri_parts.circuit import inverse_circuit
from quri_parts.qulacs.simulator import evaluate_state_to_vector

# 量子フーリエ変換(QFT)またはその逆(IQFT)を行う回路を作成
def create_qft_circuit(n, inverse=False) -> QuantumCircuit:
    # 量子ビットのインデックスは|j_{n-1} j_{n-2} ... j_0>の順番であることに注意
    circuit = QuantumCircuit(n)
    for i in reversed(range(n)): # i は n-1 から 0 まで
        circuit.add_H_gate(i)
        for j in range(i): # j は 0 から i-1 まで
            # 制御位相ゲートを4*4行列で定義
            # (本来は基礎ゲートRz,control-Rz などに分解する)
            angle = 2 * np.pi / (2 ** (i - j + 1))
            control_phase_mat = np.diag([1,1,1, np.exp(1j * angle)])
            circuit.add_UnitaryMatrix_gate([j, i], control_phase_mat)

    # 量子ビットの順序を反転
    for i in range(n // 2):
        circuit.add_SWAP_gate(i, n - i - 1)

    if inverse:
        return inverse_circuit(circuit).freeze()

    return circuit.freeze()


# ランダム配列を用いてQFT回路のデバッグ
iqft = create_qft_circuit(n_reg_qubits, inverse=True)

rng = np.random.default_rng(42) # 乱数生成器の初期化
# 複素数の乱数配列
input_np_array = rng.random(2**n_reg_qubits) + 1j * rng.random(2**n_reg_qubits)
input_np_array /= np.linalg.norm(input_np_array) # 正規化
# numpyのフーリエ変換(係数をかけて定義を合わせる)
np_result = np.fft.fft(input_np_array) / np.sqrt(2**n_reg_qubits)
# QFTの結果との比較
state = quantum_state(n_reg_qubits, vector=input_np_array, circuit=iqft)
vec = evaluate_state_to_vector(state).vector
print(np.allclose(np_result, vec)) # Trueならば等しい

True


In [7]:
def create_qpe_circuit(
    a: int, N: int, n_reg_qubits: int, L: int
    )-> QuantumCircuit:
    n_qubits = n_reg_qubits + L
    circuit = QuantumCircuit(n_qubits)

    # 1.補助量子ビットにアダマールゲートを適用
    for i in range(n_reg_qubits):
        circuit.add_H_gate(i)

    # 2.システムレジスタを|1>に初期化
    # システムレジスタのインデックスは n_reg_qubits から n_qubits-1
    circuit.add_X_gate(n_reg_qubits) # |0...01>の状態に

    # 3. i 番目の補助量子ビットを制御ビットとした
    # 制御ゲートU_a^(2^i)をシステムレジスタに作用
    target_indices = list(range(n_reg_qubits, n_qubits))

    for i in range(n_reg_qubits):
        U_a_power_matrix = create_U_a(a, N, L, power=2**i)
        # U_a^{2^i}の行列表示に，制御量子ビットを追加した行列を作成.
        # インデックスの二進数表示の左端の桁が制御量子ビットになるような行列表示で,
        # c-U_a[0y_{L-1}...y_{0}, 0x_{L-1}...x_{0}]をIに,
        # c-U_a[1y_{L-1}...y_{0}, 1x_{L-1}...x_{0}]をU_a^{2^i}にする.
        control_U_power_matrix = np.zeros((2**(L+1), 2**(L+1)), dtype=float)
        control_U_power_matrix[:2**L, :2**L] = np.eye(2**L)
        control_U_power_matrix[2**L:, :][:, 2**L:] = U_a_power_matrix
        # 制御ゲートU_a^(2^i)を追加. 制御量子ビット i は最後(左端) のインデックス
        circuit.add_UnitaryMatrix_gate(target_indices + [i], control_U_power_matrix)

    # 4. 逆量子フーリエ変換(IQFT)をカウントレジスタに適用
    # IQFTの対象は n_reg_qubits 個のレジスタ(0からn_reg_qubits-1)
    iqft = create_qft_circuit(n_reg_qubits, inverse=True)
    circuit.extend(iqft)

    return circuit

In [8]:
# 量子位相推定アルゴリズムの回路を構築
qpe_circuit = create_qpe_circuit(a, N, n_reg_qubits, L)
# 量子状態を作成
state = quantum_state(n_qubits, circuit=qpe_circuit)
# ベクトルを取得
vec = evaluate_state_to_vector(state).vector

# 補助量子ビットの観測確率を計算
probs_allbits = np.abs(vec)**2
reshaped_array = probs_allbits.reshape(2**L, 2**n_reg_qubits)
probs_regbits = reshaped_array.sum(axis=0) # システムレジスタについて周辺化

# 補助量子ビットの観測確率の高い順に10個表示
for bit in np.argsort(probs_regbits)[::-1][:10]:
    if probs_regbits[bit] < 1e-5: # 確率が非常に小さいものは無視
        continue
    print(f"--- ビット: {np.binary_repr(bit, width=n_reg_qubits)},"
          f"確率: {probs_regbits[bit]:.5f} ---")
    phase = bit / (2**n_reg_qubits)
    print(f"位相 = 2 π * {phase:.8f}")

--- ビット: 10000000,確率: 0.16669 ---
位相 = 2 π * 0.50000000
--- ビット: 00000000,確率: 0.16669 ---
位相 = 2 π * 0.00000000
--- ビット: 10101011,確率: 0.11400 ---
位相 = 2 π * 0.66796875
--- ビット: 11010101,確率: 0.11400 ---
位相 = 2 π * 0.83203125
--- ビット: 01010101,確率: 0.11400 ---
位相 = 2 π * 0.33203125
--- ビット: 00101011,確率: 0.11400 ---
位相 = 2 π * 0.16796875
--- ビット: 00101010,確率: 0.02851 ---
位相 = 2 π * 0.16406250
--- ビット: 10101010,確率: 0.02851 ---
位相 = 2 π * 0.66406250
--- ビット: 11010110,確率: 0.02851 ---
位相 = 2 π * 0.83593750
--- ビット: 01010110,確率: 0.02851 ---
位相 = 2 π * 0.33593750


In [9]:
def continued_fraction(num, max_iterations=10):
    """ num の連分数展開を計算する関数．
    連分数展開の整数[a_0, a_1, ...]のリストが出力される．
    """
    fractions = []
    current_num = float(num)

    for _ in range(max_iterations):
        # 整数部分を計算
        integer_part = int(current_num)
        fractions.append(integer_part)
        # 小数部分を計算
        decimal_part = current_num - integer_part
        # 小数部分が非常に小さい場合，終了
        if abs(decimal_part) < 1e-10:
            break
        # 次の項を計算するために逆数をとる
        current_num = 1.0 / decimal_part
    return fractions

def calculate_all_convergents(fractions):
    """ 連分数展開を表す整数のリストから，すべての有理数表示 s/r を計算し，
    (s, r)のタプルのリストを返す．
    """
    convergents = []
    # 漸化式の初期値. p[-2]=0, q[-2]=1, p[-1]=1, q[-1]=0
    p_prev_prev = 0; q_prev_prev = 1
    p_prev = 1; q_prev = 0

    for i, a_i in enumerate(fractions):
        # 現在の分子p_i と分母q_i を計算
        p_current = a_i * p_prev + p_prev_prev
        q_current = a_i * q_prev + q_prev_prev
        convergents.append((p_current, q_current))
        # 次のイテレーションのために値を更新
        p_prev_prev = p_prev; q_prev_prev = q_prev
        p_prev = p_current; q_prev = q_current

    return convergents

# 例： pi の連分数展開
pi_fractions = continued_fraction(np.pi, max_iterations=10)
print(pi_fractions)
# 各次数の近似を計算
pi_convergents = calculate_all_convergents(pi_fractions)
print(f"連分数近似:")
for s, r in pi_convergents:
    print(f" {s}/{r} = {s / r:.10f}")

[3, 7, 15, 1, 292, 1, 1, 1, 2, 1]
連分数近似:
 3/1 = 3.0000000000
 22/7 = 3.1428571429
 333/106 = 3.1415094340
 355/113 = 3.1415929204
 103993/33102 = 3.1415926530
 104348/33215 = 3.1415926539
 208341/66317 = 3.1415926535
 312689/99532 = 3.1415926536
 833719/265381 = 3.1415926536
 1146408/364913 = 3.1415926536


書籍版からの変更点：`約数発見\(^o^)/` が文法エラーになってしまうのを防ぐために、文字列の冒頭に`r`を足しています。

In [10]:
for bit in np.argsort(probs_regbits)[::-1][:5]: # 確率の高い順に5つ調べる
    if probs_regbits[bit] < 1e-5: # 確率が非常に小さいものは無視
        continue
    print(f"--- ビット: {np.binary_repr(bit, width=n_reg_qubits)},"
          f"確率: {probs_regbits[bit]:.5f} ---")
    phase = bit / (2**n_reg_qubits)
    print(f"位相 = 2 π * {phase:.8f}")
    fractions = continued_fraction(phase, max_iterations=10)
    convergents = calculate_all_convergents(fractions)
    for s_cand, r_cand in convergents:
        a_to_r_cand = pow(a, r_cand, N)
        print(f" 候補分数 {s_cand}/{r_cand} の場合: a^r mod N = {a_to_r_cand}")
        if a_to_r_cand == 1 and r_cand % 2 == 0:
            a_to_r_half = pow(a, r_cand // 2, N)
            if a_to_r_half == 1 or a_to_r_half == N - 1:
                print(" [a^(r/2) mod N = 1 または -1 の場合は無視]")
                continue
            divisor_1 = math.gcd(a_to_r_half - 1, N)
            print(f"  gcd(a^(r/2) - 1, N) = {divisor_1}")
            if N % divisor_1 == 0:
                print(r"  約数発見 \(^o^)/ ")
            divisor_2 = math.gcd(a_to_r_half + 1, N)
            print(f"  gcd(a^(r/2) + 1, N) = {divisor_2}")
            if N % divisor_2 == 0:
                print(r"  約数発見 \(^o^)/ ")

--- ビット: 10000000,確率: 0.16669 ---
位相 = 2 π * 0.50000000
 候補分数 0/1 の場合: a^r mod N = 2
 候補分数 1/2 の場合: a^r mod N = 4
--- ビット: 00000000,確率: 0.16669 ---
位相 = 2 π * 0.00000000
 候補分数 0/1 の場合: a^r mod N = 2
--- ビット: 10101011,確率: 0.11400 ---
位相 = 2 π * 0.66796875
 候補分数 0/1 の場合: a^r mod N = 2
 候補分数 1/1 の場合: a^r mod N = 2
 候補分数 2/3 の場合: a^r mod N = 8
 候補分数 169/253 の場合: a^r mod N = 2
 候補分数 171/256 の場合: a^r mod N = 16
--- ビット: 11010101,確率: 0.11400 ---
位相 = 2 π * 0.83203125
 候補分数 0/1 の場合: a^r mod N = 2
 候補分数 1/1 の場合: a^r mod N = 2
 候補分数 4/5 の場合: a^r mod N = 11
 候補分数 5/6 の場合: a^r mod N = 1
  gcd(a^(r/2) - 1, N) = 7
  約数発見 \(^o^)/ 
  gcd(a^(r/2) + 1, N) = 3
  約数発見 \(^o^)/ 
 候補分数 104/125 の場合: a^r mod N = 11
 候補分数 213/256 の場合: a^r mod N = 16
--- ビット: 01010101,確率: 0.11400 ---
位相 = 2 π * 0.33203125
 候補分数 0/1 の場合: a^r mod N = 2
 候補分数 1/3 の場合: a^r mod N = 8
 候補分数 85/256 の場合: a^r mod N = 16
